In [16]:
import os
import subprocess

In [ ]:
from concurrent.futures import ThreadPoolExecutor
max_processes = 4

cpu_count = os.cpu_count() or 1
threads = max(2, cpu_count // 3)

In [18]:
input_file = "/home/led/HLA_temp/Datas/hac_dir/"

input_files = [f for f in os.listdir(input_file) if f.endswith('.fastq.gz')]

In [19]:
output_file = "/home/led/temp/development/seqkit/"
new_names = "raw"
output_file = os.path.join(output_file, new_names)

os.makedirs(output_file, exist_ok=True)

In [ ]:
sequence_formats = (
    ".fa", ".fasta", ".fas", ".fsa",
    ".fna", ".ffn", ".faa", ".frn",
    ".fq", ".fastq"
)

compression_formats = ("", ".gz", ".xz", ".zst", ".bz2", ".lz4")

file_formats = []

for sequence in sequence_formats:
    for compression in compression_formats:
        file_formats.append(sequence + compression)

input_files = [
    f for f in os.listdir(input_file)
    if f.lower().endswith(tuple(file_formats))
    and os.path.isfile(os.path.join(input_file, f))
]

In [21]:
for i in input_files: 
    
    def run_seqkit(i):

        for suffix in file_formats:
            if i.lower().endswith(suffix):
                sample_id = i[:-len(suffix)]
                break

        input_path = os.path.join(input_file, i)

        output_path = os.path.join(output_file, f"{sample_id}.stats.txt")

        cmd = [
            "seqkit", "stats",
            input_path,
            "-o", output_path,
            "-a",
            "-j", str(threads)
        ]

        subprocess.run(cmd, check=True)

        return sample_id

with ThreadPoolExecutor(max_workers=max_processes) as executor:

    for sample_id in executor.map(run_seqkit, input_files):
        print(f"{sample_id} 统计完成")


3 统计完成
5 统计完成
2 统计完成
4 统计完成
1 统计完成
